In [0]:
from pyspark.sql.functions import to_timestamp, to_utc_timestamp, col

# Access the bronze table from catalog
bronze_dataframe = spark.read.table("weather_project.north_texas_weather.bronze_hourly_multi_city")

# Rename columns for better readability
silver_dataframe = bronze_dataframe \
    .withColumnRenamed("temperature_2m", "temperature_celsius") \
    .withColumnRenamed("precipitation", "precipitation_inches")

# Convert time to UTC (good practice to prevent time conversion bugs when servers operate across different locations)
# First convert time to a timestamp, then convert to UTC
silver_dataframe = silver_dataframe.withColumn("local_time", to_timestamp(col("time"), "yyyy-MM-dd'T'HH:mm")) \
                                    .withColumn("utc_time", to_utc_timestamp(col("local_time"), "America/Chicago")) \
                                    .drop("time", "local_time")

# Ignore rows with identical times (duplicates). Use UTC time for more precision
silver_dataframe = silver_dataframe.dropDuplicates(["city", "utc_time"])

# Rows without temperature are useless so they are dropped
silver_dataframe = silver_dataframe.na.drop(subset=["temperature_celsius"])

# Null precipitation usually means no precipitation, so it's replaced with 0
silver_dataframe = silver_dataframe.na.fill(0, ["precipitation_inches"])

# Convert temperature from Celsius to Fahrenheit
silver_dataframe = silver_dataframe.withColumn("temperature_fahrenheit", (col("temperature_celsius") * 9 / 5) + 32)

# Validate ranges
silver_dataframe = silver_dataframe.filter(col("temperature_fahrenheit") >= -100) \
                                  .filter(col("temperature_fahrenheit") <= 200) \
                                  .filter(col("precipitation_inches") >= 0) \
                                  .filter(col("precipitation_inches") <= 300)

# Write table to Delta Lake
# .format("delta") writes the data in Delta Lake format, which adds several capabilities to the data, such as time travel and schema enforcement
# .mode("overwrite") overwrites the table if it already exists
silver_dataframe.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_project.north_texas_weather.silver_hourly_multi_city")

print("Successfully built and saved the multi-city Silver table!")